In [3]:
# ============================================================
# 03_feature_engineering.ipynb | CELL 1 — Feature Matrix (Fixed)
#
# FIX: Removed all_side_effects TF-IDF (was causing data leakage)
# Features now: name TF-IDF + uses TF-IDF + structured only
# Target      : severity (MILD/MODERATE/CRITICAL)
# Model learns: Given drug name + medical use + drug class
#               → predict ADR severity profile
# ============================================================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix, save_npz
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────
BASE = "/teamspace/studios/this_studio/data/drug_reviews/"
SAVE = os.path.join(BASE, "features/")
os.makedirs(SAVE, exist_ok=True)

# ── Load Train / Test ─────────────────────────────────────────
train_df = pd.read_csv(os.path.join(BASE, "train.csv"))
test_df  = pd.read_csv(os.path.join(BASE, "test.csv"))

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}\n")

# ── Step 1: TF-IDF on drug name ───────────────────────────────
# Captures drug family patterns (e.g. "tablet", "syrup", brand names)
tfidf_name = TfidfVectorizer(
    max_features = 300,
    ngram_range  = (1, 2),
    min_df       = 2,
    sublinear_tf = True,
    stop_words   = "english"
)

X_train_name = tfidf_name.fit_transform(train_df["name"].fillna(""))
X_test_name  = tfidf_name.transform(test_df["name"].fillna(""))

print(f"TF-IDF name   → train: {X_train_name.shape} | test: {X_test_name.shape}")

# ── Step 2: TF-IDF on all_uses ───────────────────────────────
# Captures what the drug treats — strong signal for severity class
tfidf_use = TfidfVectorizer(
    max_features = 200,
    ngram_range  = (1, 2),
    min_df       = 2,
    sublinear_tf = True,
    stop_words   = "english"
)

X_train_use = tfidf_use.fit_transform(train_df["all_uses"].fillna(""))
X_test_use  = tfidf_use.transform(test_df["all_uses"].fillna(""))

print(f"TF-IDF uses   → train: {X_train_use.shape} | test: {X_test_use.shape}")

# ── Step 3: Structured Features ───────────────────────────────
# se_count: number of side effects (independent proxy for severity)
# habit_forming, therapeutic/chemical/action class encodings
STRUCT_COLS = [
    "se_count",
    "habit_forming",
    "Therapeutic Class_enc",
    "Chemical Class_enc",
    "Action Class_enc"
]

X_train_struct = csr_matrix(train_df[STRUCT_COLS].values.astype(float))
X_test_struct  = csr_matrix(test_df[STRUCT_COLS].values.astype(float))

print(f"Structured    → train: {X_train_struct.shape} | test: {X_test_struct.shape}")

# ── Step 4: Combine All Features ──────────────────────────────
X_train = hstack([X_train_name, X_train_use, X_train_struct])
X_test  = hstack([X_test_name,  X_test_use,  X_test_struct])

y_train = train_df["label"].values
y_test  = test_df["label"].values

print(f"\nFinal feature matrix (NO leakage):")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape} | classes: {np.unique(y_train)}")
print(f"  y_test  : {y_test.shape}  | classes: {np.unique(y_test)}")

# ── Step 5: Save Everything ───────────────────────────────────
save_npz(os.path.join(SAVE, "X_train.npz"), X_train)
save_npz(os.path.join(SAVE, "X_test.npz"),  X_test)
np.save(os.path.join(SAVE, "y_train.npy"),  y_train)
np.save(os.path.join(SAVE, "y_test.npy"),   y_test)

with open(os.path.join(SAVE, "tfidf_name.pkl"), "wb") as f: pickle.dump(tfidf_name, f)
with open(os.path.join(SAVE, "tfidf_use.pkl"),  "wb") as f: pickle.dump(tfidf_use,  f)

print(f"\nSaved to: {SAVE}")
print("  X_train.npz, X_test.npz")
print("  y_train.npy, y_test.npy")
print("  tfidf_name.pkl, tfidf_use.pkl")

# ── Step 6: Sanity Check — Top TF-IDF Terms ───────────────────
print("\nTop 15 TF-IDF terms from drug name:")
terms  = tfidf_name.get_feature_names_out()
scores = X_train_name.mean(axis=0).A1
for i in scores.argsort()[::-1][:15]:
    print(f"  {terms[i]:35s}  {scores[i]:.4f}")

print("\nTop 15 TF-IDF terms from uses:")
terms  = tfidf_use.get_feature_names_out()
scores = X_train_use.mean(axis=0).A1
for i in scores.argsort()[::-1][:15]:
    print(f"  {terms[i]:35s}  {scores[i]:.4f}")

Train : (198574, 12)
Test  : (49644, 12)

TF-IDF name   → train: (198574, 300) | test: (49644, 300)
TF-IDF uses   → train: (198574, 200) | test: (49644, 200)
Structured    → train: (198574, 5) | test: (49644, 5)

Final feature matrix (NO leakage):
  X_train : (198574, 505)
  X_test  : (49644, 505)
  y_train : (198574,) | classes: [0 1 2]
  y_test  : (49644,)  | classes: [0 1 2]

Saved to: /teamspace/studios/this_studio/data/drug_reviews/features/
  X_train.npz, X_test.npz
  y_train.npy, y_test.npy
  tfidf_name.pkl, tfidf_use.pkl

Top 15 TF-IDF terms from drug name:
  tablet                               0.1744
  injection                            0.0604
  syrup                                0.0491
  500mg                                0.0387
  capsule                              0.0375
  5mg                                  0.0292
  10mg                                 0.0286
  200mg                                0.0280
  100mg                                0.0271
  500mg tablet